# 13 — Combined: Regex + 4-bit Qwen Agent + Ontology Normalization

**Pipeline order (highest confidence wins):**

1. **PRIDE API** → organism, tissue, disease, instrument per PXD
2. **Regex from paper text** → protocol params (tolerances, gradient, enzyme, label, mods)
3. **4-bit Qwen 7B agent** → harder semantic fields (disease nuance, cell type, genotype, strain)
4. **Filename parser** → per-file: fraction, replicate, condition
5. **Ontology normalization** → every value mapped to canonical PSI-MS/UBERON/UNIMOD
6. **Majority fallback** → high-frequency columns still empty

Regex fills protocol columns reliably. LLM fills semantic columns regex can't. Neither alone is enough.

## 0. Install bitsandbytes (first time only)

In [ ]:
import subprocess, sys
try:
    import bitsandbytes
    print(f'bitsandbytes {bitsandbytes.__version__} already installed')
except ImportError:
    print('Installing bitsandbytes...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'bitsandbytes', '-q'])
    print('Done.')

## 1. Imports and paths

In [ ]:
import os, re, json, time, warnings
from pathlib import Path
from collections import defaultdict, Counter

import requests
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
)
warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle').exists()

if IS_KAGGLE:
    MODEL_ID        = str(Path('/kaggle/input/qwen2-5-7b-instruct'))
    DATA_PATH       = Path('/kaggle/input/harmonizing-the-data-of-your-data')
    OUT_PATH        = Path('/kaggle/working/submission_combined.csv')
    USE_4BIT        = False   # T4 has 16 GB
    PRIDE_TIMEOUT   = 0       # no internet on Kaggle
else:
    ROOT            = Path.cwd().parent
    MODEL_ID        = 'Qwen/Qwen2.5-7B-Instruct'
    DATA_PATH       = ROOT / 'data'
    OUT_PATH        = ROOT / 'outputs' / 'submission_combined.csv'
    USE_4BIT        = True    # GTX 1060 needs 4-bit
    PRIDE_TIMEOUT   = 15      # internet available locally

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
SAMPLE_SUB     = DATA_PATH / 'SampleSubmission.csv'
TRAIN_SDRF_DIR = DATA_PATH / 'TrainingSDRFs'
if not TRAIN_SDRF_DIR.exists():
    TRAIN_SDRF_DIR = DATA_PATH / 'Training_SDRFs' / 'HarmonizedFiles'

_pubtext_candidates = [
    DATA_PATH / 'Test_PubText' / 'Test PubText',
    DATA_PATH / 'Test_PubText',
    DATA_PATH / 'TestPubText',
    DATA_PATH / 'Test PubText',
]
PUBTEXT_DIR = next((p for p in _pubtext_candidates if p.exists()),
                   _pubtext_candidates[0])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device    : {device}')
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU       : {torch.cuda.get_device_name(0)} ({total:.1f} GB)')
print(f'Model     : {MODEL_ID}')
print(f'4-bit     : {USE_4BIT}')
print(f'PRIDE API : {"disabled (Kaggle)" if not PRIDE_TIMEOUT else "enabled"}')
print(f'PubText   : {PUBTEXT_DIR} — exists: {PUBTEXT_DIR.exists()}')

## 2. Ontology dictionaries

In [ ]:
INSTRUMENT_ONT = {
    'q exactive hf-x':        'AC=MS:1003027;NT=Q Exactive HF-X',
    'q exactive hf':          'AC=MS:1002523;NT=Q Exactive HF',
    'q exactive plus':        'AC=MS:1002634;NT=Q Exactive Plus',
    'q-exactive plus':        'AC=MS:1002634;NT=Q Exactive Plus',
    'q exactive':             'AC=MS:1001911;NT=Q Exactive',
    'qexactive':              'AC=MS:1001911;NT=Q Exactive',
    'orbitrap astral':        'AC=MS:1003378;NT=Orbitrap Astral',
    'orbitrap fusion lumos':  'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
    'fusion lumos':           'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
    'orbitrap fusion':        'AC=MS:1002416;NT=Orbitrap Fusion',
    'orbitrap eclipse':       'AC=MS:1003029;NT=Orbitrap Eclipse',
    'orbitrap exploris 480':  'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'exploris 480':           'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'ltq orbitrap velos':     'AC=MS:1001742;NT=LTQ Orbitrap Velos',
    'ltq orbitrap elite':     'AC=MS:1001910;NT=LTQ Orbitrap Elite',
    'ltq orbitrap xl':        'AC=MS:1000556;NT=LTQ Orbitrap XL',
    'ltq orbitrap':           'AC=MS:1000449;NT=LTQ Orbitrap',
    'timstof pro 2':          'AC=MS:1003474;NT=timsTOF Pro 2',
    'timstof pro':            'AC=MS:1003231;NT=timsTOF Pro',
    'timstof':                'AC=MS:1002817;NT=timsTOF',
    'triple tof 6600':        'AC=MS:1000931;NT=TripleTOF 6600',
    'triple tof 5600':        'AC=MS:1000931;NT=TripleTOF 5600',
    'synapt g2-si':           'AC=MS:1002726;NT=Synapt G2-Si',
    'synapt g2':              'AC=MS:1002726;NT=Synapt G2-Si',
    'impact ii':              'AC=MS:1002817;NT=impact II',
    'velos pro':              'AC=MS:1001909;NT=LTQ Velos Pro',
}
TISSUE_ONT = {
    'blood plasma': 'NT=blood plasma;AC=UBERON:0001969',
    'plasma': 'NT=blood plasma;AC=UBERON:0001969',
    'blood serum': 'NT=blood serum;AC=UBERON:0001977',
    'serum': 'NT=blood serum;AC=UBERON:0001977',
    'whole blood': 'NT=blood;AC=UBERON:0000178',
    'blood': 'NT=blood;AC=UBERON:0000178',
    'cerebrospinal fluid': 'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'csf': 'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'urine': 'NT=urine;AC=UBERON:0001088',
    'brain': 'NT=brain;AC=UBERON:0000955',
    'prefrontal cortex': 'NT=prefrontal cortex;AC=UBERON:0000451',
    'frontal cortex': 'NT=frontal cortex;AC=UBERON:0001870',
    'cerebral cortex': 'NT=cerebral cortex;AC=UBERON:0000956',
    'cortex': 'NT=cerebral cortex;AC=UBERON:0000956',
    'hippocampus': 'NT=hippocampal formation;AC=UBERON:0002421',
    'cerebellum': 'NT=cerebellum;AC=UBERON:0002037',
    'liver': 'NT=liver;AC=UBERON:0002107',
    'lung': 'NT=lung;AC=UBERON:0002048',
    'heart': 'NT=heart;AC=UBERON:0000948',
    'kidney': 'NT=kidney;AC=UBERON:0002113',
    'pancreas': 'NT=pancreas;AC=UBERON:0001264',
    'colon': 'NT=colon;AC=UBERON:0001155',
    'spleen': 'NT=spleen;AC=UBERON:0002106',
    'bone marrow': 'NT=bone marrow;AC=UBERON:0002371',
    'adipose tissue': 'NT=adipose tissue;AC=UBERON:0001013',
    'adipose': 'NT=adipose tissue;AC=UBERON:0001013',
    'skeletal muscle': 'NT=skeletal muscle;AC=UBERON:0001134',
    'muscle': 'NT=skeletal muscle;AC=UBERON:0001134',
    'retina': 'NT=retina;AC=UBERON:0000966',
    'prostate': 'NT=prostate gland;AC=UBERON:0002367',
    'breast': 'NT=breast;AC=UBERON:0000310',
    'ovary': 'NT=ovary;AC=UBERON:0000992',
    'pbmc': 'NT=peripheral blood mononuclear cell;AC=CL:0000057',
    'platelet': 'NT=platelet;AC=CL:0000233',
    'exosome': 'NT=extracellular vesicle;AC=GO:0061695',
}
ORGANISM_ONT = {
    'homo sapiens': '9606 (Homo sapiens)', 'human': '9606 (Homo sapiens)',
    'humans': '9606 (Homo sapiens)', 'patient': '9606 (Homo sapiens)',
    'mus musculus': '10090 (Mus musculus)', 'mouse': '10090 (Mus musculus)',
    'mice': '10090 (Mus musculus)', 'murine': '10090 (Mus musculus)',
    'rattus norvegicus': '10116 (Rattus norvegicus)', 'rat': '10116 (Rattus norvegicus)',
    'saccharomyces cerevisiae': '4932 (Saccharomyces cerevisiae)',
    'yeast': '4932 (Saccharomyces cerevisiae)',
    'escherichia coli': '562 (Escherichia coli)',
    'e. coli': '562 (Escherichia coli)', 'e.coli': '562 (Escherichia coli)',
    'danio rerio': '7955 (Danio rerio)', 'zebrafish': '7955 (Danio rerio)',
    'drosophila melanogaster': '7227 (Drosophila melanogaster)',
    'arabidopsis thaliana': '3702 (Arabidopsis thaliana)',
    'sus scrofa': '9823 (Sus scrofa)', 'pig': '9823 (Sus scrofa)',
    'bos taurus': '9913 (Bos taurus)', 'bovine': '9913 (Bos taurus)',
    'gallus gallus': '9031 (Gallus gallus)', 'chicken': '9031 (Gallus gallus)',
    'macaca mulatta': '9544 (Macaca mulatta)', 'rhesus': '9544 (Macaca mulatta)',
}
CLEAVAGE_ONT = {
    'trypsin/lysc': 'AC=MS:1001251;NT=Trypsin',
    'trypsin/lys-c': 'AC=MS:1001251;NT=Trypsin',
    'trypsin': 'AC=MS:1001251;NT=Trypsin',
    'tryps': 'AC=MS:1001251;NT=Trypsin',
    'tryptic': 'AC=MS:1001251;NT=Trypsin',
    'lys-c': 'AC=MS:1001255;NT=Lys-C', 'lysc': 'AC=MS:1001255;NT=Lys-C',
    'lysyl endopeptidase': 'AC=MS:1001255;NT=Lys-C',
    'glu-c': 'AC=MS:1001917;NT=Glu-C', 'gluc': 'AC=MS:1001917;NT=Glu-C',
    'v8 protease': 'AC=MS:1001917;NT=Glu-C',
    'chymotrypsin': 'AC=MS:1001306;NT=Chymotrypsin',
    'asp-n': 'AC=MS:1001267;NT=Asp-N', 'aspn': 'AC=MS:1001267;NT=Asp-N',
    'arg-c': 'AC=MS:1001303;NT=Arg-C',
    'elastase': 'AC=MS:1001915;NT=Elastase',
    'pepsin': 'AC=MS:1001940;NT=Pepsin',
}
LABEL_ONT = {
    'label-free': 'AC=MS:1002038;NT=label free sample',
    'label free': 'AC=MS:1002038;NT=label free sample',
    'lfq': 'AC=MS:1002038;NT=label free sample',
    'unlabeled': 'AC=MS:1002038;NT=label free sample',
    'tmt18plex': 'AC=MS:1003999;NT=TMT18plex',
    'tmt16plex': 'AC=MS:1003998;NT=TMT16plex',
    'tmtpro': 'AC=MS:1003998;NT=TMT16plex',
    'tmt pro': 'AC=MS:1003998;NT=TMT16plex',
    'tmt11plex': 'AC=MS:1002454;NT=TMT11plex',
    'tmt10plex': 'AC=MS:1002454;NT=TMT10plex',
    'tmt6plex': 'AC=MS:1002453;NT=TMT6plex',
    'tmt6': 'AC=MS:1002453;NT=TMT6plex',
    'tmt': 'AC=MS:1002453;NT=TMT6plex',
    'itraq8plex': 'AC=MS:1002519;NT=iTRAQ8plex',
    'itraq4plex': 'AC=MS:1001985;NT=iTRAQ4plex',
    'itraq': 'AC=MS:1001985;NT=iTRAQ4plex',
    'silac': 'AC=MS:1002791;NT=SILAC',
    'stable isotope labeling': 'AC=MS:1002791;NT=SILAC',
    'dimethyl': 'AC=MS:1002457;NT=Dimethyl',
    'tandem mass tag': 'AC=MS:1002453;NT=TMT6plex',
}
FRAG_ONT = {
    'hcd': 'AC=MS:1002481;NT=HCD',
    'higher-energy collisional dissociation': 'AC=MS:1002481;NT=HCD',
    'higher energy collisional dissociation': 'AC=MS:1002481;NT=HCD',
    'cid': 'AC=MS:1001880;NT=CID',
    'collision-induced dissociation': 'AC=MS:1001880;NT=CID',
    'etd': 'AC=MS:1001526;NT=ETD',
    'electron transfer dissociation': 'AC=MS:1001526;NT=ETD',
    'ecd': 'AC=MS:1001872;NT=ECD',
    'uvpd': 'AC=MS:1003246;NT=UVPD',
    'ethcd': 'AC=MS:1002686;NT=EThcD',
}
MOD_ONT = {
    'carbamidomethyl': 'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed',
    'carbamidomethylation': 'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed',
    'iodoacetamide': 'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed',
    'oxidation': 'NT=Oxidation;AC=UNIMOD:35;TA=M;MT=Variable',
    'oxidized methionine': 'NT=Oxidation;AC=UNIMOD:35;TA=M;MT=Variable',
    'phosphorylation': 'NT=Phospho;AC=UNIMOD:21;TA=S,T,Y;MT=Variable',
    'phospho': 'NT=Phospho;AC=UNIMOD:21;TA=S,T,Y;MT=Variable',
    'acetylation': 'NT=Acetyl;AC=UNIMOD:1;TA=K;MT=Variable',
    'acetyl': 'NT=Acetyl;AC=UNIMOD:1;TA=K;MT=Variable',
    'ubiquitination': 'NT=GlyGly;AC=UNIMOD:121;TA=K;MT=Variable',
    'diglycine': 'NT=GlyGly;AC=UNIMOD:121;TA=K;MT=Variable',
    'methylation': 'NT=Methyl;AC=UNIMOD:34;TA=K,R;MT=Variable',
    'deamidation': 'NT=Deamidated;AC=UNIMOD:7;TA=N,Q;MT=Variable',
    'deamidated': 'NT=Deamidated;AC=UNIMOD:7;TA=N,Q;MT=Variable',
    'succinylation': 'NT=Succinyl;AC=UNIMOD:64;TA=K;MT=Variable',
    'crotonylation': 'NT=Crotonyl;AC=UNIMOD:1363;TA=K;MT=Variable',
    'glutarylation': 'NT=Glutaryl;AC=UNIMOD:1848;TA=K;MT=Variable',
    'malonylation': 'NT=Malonyl;AC=UNIMOD:747;TA=K;MT=Variable',
    'tmt6plex': 'NT=TMT6plex;AC=UNIMOD:737;TA=K;MT=Fixed',
    'tmt label': 'NT=TMT6plex;AC=UNIMOD:737;TA=K;MT=Fixed',
}
REDUCTION_ONT = {
    'dtt': 'AC=MS:1000578;NT=DTT',
    'dithiothreitol': 'AC=MS:1000578;NT=DTT',
    'tcep': 'AC=MS:1001135;NT=TCEP',
    'tris(2-carboxyethyl)phosphine': 'AC=MS:1001135;NT=TCEP',
    'beta-mercaptoethanol': 'AC=MS:1000382;NT=beta-mercaptoethanol',
}
ALKYLATION_ONT = {
    'iodoacetamide': 'AC=PRIDE:0000126;NT=Iodoacetamide',
    'iaa': 'AC=PRIDE:0000126;NT=Iodoacetamide',
    'chloroacetamide': 'AC=PRIDE:0000126;NT=Chloroacetamide',
    'caa': 'AC=PRIDE:0000126;NT=Chloroacetamide',
    'n-ethylmaleimide': 'AC=PRIDE:0000459;NT=N-ethylmaleimide',
}

def normalize(raw, ontology):
    if not raw: return None
    s = str(raw).lower().strip()
    for key in sorted(ontology, key=len, reverse=True):
        if key in s: return ontology[key]
    return None

def normalize_instrument(raw):
    if not raw: return None
    s = str(raw)
    ac = re.search(r'AC=(MS:\d+)', s)
    nt = re.search(r'NT=([^;]+)', s)
    if ac and nt:
        return f'AC={ac.group(1).strip()};NT={nt.group(1).strip()}'
    return normalize(raw, INSTRUMENT_ONT)

print('Ontology dictionaries loaded.')
print(f'  Instruments:{len(INSTRUMENT_ONT)} Tissues:{len(TISSUE_ONT)} '
      f'Mods:{len(MOD_ONT)} Labels:{len(LABEL_ONT)}')

## 3. Load Qwen2.5-7B (4-bit locally, float16 on Kaggle)

In [ ]:
print(f'Loading tokenizer: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16,
        device_map='auto', trust_remote_code=True,
    )

model.eval()
print('Model loaded.')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {used:.1f} / {total:.1f} GB ({total-used:.1f} GB free)')

## 4. Load test data and training vocabulary

In [ ]:
SECTIONS = ['TITLE','ABSTRACT','METHODS','MATERIALS AND METHODS',
            'EXPERIMENTAL','SAMPLE PREPARATION','MASS SPECTROMETRY',
            'LC-MS','PROTEIN DIGESTION','DATA ACQUISITION']
METHOD_KWS = ['method','material','protocol','digest','spectr',
              'chromat','prep','enrichment','experimental']

def get_text(pub_dict):
    parts = []
    for key in SECTIONS:
        val = pub_dict.get(key, '')
        if isinstance(val, list): val = ' '.join(str(x) for x in val)
        if val.strip(): parts.append(val.strip())
    for key, val in pub_dict.items():
        if key.upper() in SECTIONS: continue
        if any(kw in key.lower() for kw in METHOD_KWS):
            if isinstance(val, list): val = ' '.join(str(x) for x in val)
            if val.strip(): parts.append(val.strip())
    for key in ['ABSTRACT','TITLE']:
        val = pub_dict.get(key, '')
        if isinstance(val, list): val = ' '.join(str(x) for x in val)
        if val.strip(): parts.append(val.strip())
    return ' '.join(parts)

sample_sub  = pd.read_csv(SAMPLE_SUB, dtype=str)
target_cols = [c for c in sample_sub.columns
               if c not in ('ID','PXD','Raw Data File')]

pxd_to_raws = {}
for _, row in sample_sub.iterrows():
    pxd_to_raws.setdefault(row['PXD'], []).append(row['Raw Data File'])

test_docs = {}
if PUBTEXT_DIR.exists():
    for fp in sorted(PUBTEXT_DIR.glob('*.json')):
        pxd = fp.stem.split('_')[0]
        try:
            d = json.loads(fp.read_text(encoding='utf-8', errors='replace'))
            if d: test_docs[pxd] = d
        except: pass

# Training vocab for majority fallback
col_counters = {col: Counter() for col in target_cols}
train_files  = []
if TRAIN_SDRF_DIR.exists():
    train_files = list(TRAIN_SDRF_DIR.glob('*.tsv')) + list(TRAIN_SDRF_DIR.glob('*.csv'))

for fp in train_files:
    sep = '\t' if fp.suffix == '.tsv' else ','
    try:
        df = pd.read_csv(fp, low_memory=False, sep=sep)
    except: continue
    for col in target_cols:
        base = re.sub(r'\.\d+$', '', col)
        mc = col if col in df.columns else base if base in df.columns else None
        if mc:
            vals = df[mc].dropna().astype(str)
            vals = vals[~vals.str.lower().isin(['not applicable','na',''])]
            col_counters[col].update(vals.tolist())

global_modes = {}
for col in target_cols:
    if col_counters[col]:
        global_modes[col] = col_counters[col].most_common(1)[0][0]

print(f'Test docs   : {len(test_docs)}')
print(f'Train SDRFs : {len(train_files)}')
print(f'Target cols : {len(target_cols)}')
for pxd, d in test_docs.items():
    print(f'  {pxd}: {len(get_text(d)):,} chars')

## 5. PRIDE API (runs locally, skipped on Kaggle)

In [ ]:
http_session = requests.Session()
http_session.headers.update({'User-Agent': 'SDRF-Combined/1.0'})

def fetch_pride(pxd):
    if not PRIDE_TIMEOUT: return {}
    try:
        r = http_session.get(
            f'https://www.ebi.ac.uk/pride/ws/archive/v2/projects/{pxd}',
            timeout=PRIDE_TIMEOUT
        )
        if r.status_code != 200: return {}
        d = r.json()
        out = defaultdict(list)
        for o in d.get('organisms',[]):
            name = o.get('name','')
            if name:
                norm = normalize(name, ORGANISM_ONT)
                out['Characteristics[Organism]'].append(norm or name.strip())
        for op in (d.get('organisms_part') or d.get('tissues') or []):
            name = op.get('name',''); acc = op.get('accession','')
            if name and name.lower() not in ('not available','n/a',''):
                norm = normalize(name, TISSUE_ONT)
                val = norm or (f'NT={name};AC={acc}' if acc else f'NT={name}')
                out['Characteristics[OrganismPart]'].append(val)
        for dis in d.get('diseases',[]):
            name = dis.get('name','')
            if name and name.lower() not in ('not available','n/a','none','normal',''):
                out['Characteristics[Disease]'].append(name)
        for inst in d.get('instruments',[]):
            name = inst.get('name',''); acc = inst.get('accession','')
            if name:
                norm = normalize_instrument(name)
                out['Comment[Instrument]'].append(
                    norm or (f'AC={acc};NT={name}' if acc else name)
                )
        for qm in d.get('quantification_methods',[]):
            name = qm.get('name','')
            if name:
                norm = normalize(name, LABEL_ONT)
                out['Characteristics[Label]'].append(norm or name)
        return {k: list(dict.fromkeys(v)) for k,v in out.items() if v}
    except Exception as e:
        print(f'  PRIDE error {pxd}: {e}')
        return {}

print('PRIDE fetcher defined.')
print(f'  Will fetch: {"YES" if PRIDE_TIMEOUT else "NO (Kaggle offline)"}')

## 6. Regex extractor

Extracts protocol parameters from paper text. Regex wins on:
precursor/fragment tolerances, gradient time, flow rate, missed cleavages,
acquisition method (DDA/DIA), ionization type, instrument name, enzyme,
organism, tissue, modification keywords, reduction/alkylation reagents.

These are keyword-matching tasks — regex is faster and more reliable than LLM here.

In [ ]:
_NEG = r'(?<!without\s)(?<!no\s)(?<!not\s)'
_CLINICAL = re.compile(
    r'\b(patient|cohort|biopsy|tumor|tumour|cancer|carcinoma|malignant|'
    r'diagnosed|clinical|disease|healthy\s+(?:control|donor)|specimen|'
    r'hospital|surgical|resection)\b', re.I
)

def regex_extract(pub_dict):
    text     = get_text(pub_dict)
    text_low = text.lower()
    out = defaultdict(list)

    def add(col, val):
        if val and val not in out[col]: out[col].append(val)

    # Organism
    for pat, key in [
        (re.compile(r'\b(homo\s+sapiens|human(?:s)?)\b', re.I), 'human'),
        (re.compile(r'\b(mus\s+musculus|mouse|mice|murine)\b', re.I), 'mouse'),
        (re.compile(r'\b(rattus\s+norvegicus|rat(?:s)?)\b', re.I), 'rat'),
        (re.compile(r'\b(saccharomyces\s+cerevisiae|(?<!\w)yeast)\b', re.I), 'yeast'),
        (re.compile(r'\b(escherichia\s+coli|e\.?\s*coli)\b', re.I), 'e. coli'),
        (re.compile(r'\b(danio\s+rerio|zebrafish)\b', re.I), 'zebrafish'),
        (re.compile(r'\b(drosophila\s+melanogaster)\b', re.I), 'drosophila melanogaster'),
        (re.compile(r'\b(sus\s+scrofa|porcine)\b', re.I), 'pig'),
        (re.compile(r'\b(bos\s+taurus|bovine)\b', re.I), 'bovine'),
        (re.compile(r'\b(gallus\s+gallus|chicken)\b', re.I), 'chicken'),
        (re.compile(r'\b(arabidopsis\s+thaliana)\b', re.I), 'arabidopsis thaliana'),
        (re.compile(r'\b(macaca\s+mulatta|rhesus\s+macaque)\b', re.I), 'macaca mulatta'),
    ]:
        if pat.search(text):
            norm = ORGANISM_ONT.get(key)
            if norm: add('Characteristics[Organism]', norm)

    # OrganismPart
    for tissue in sorted(TISSUE_ONT, key=len, reverse=True):
        if re.search(r'\b' + re.escape(tissue) + r'\b', text_low):
            add('Characteristics[OrganismPart]', TISSUE_ONT[tissue])

    # CleavageAgent
    for enzyme in sorted(CLEAVAGE_ONT, key=len, reverse=True):
        pat = re.compile(_NEG + r'\b' + re.escape(enzyme) + r'\b', re.I)
        if pat.search(text_low):
            add('Characteristics[CleavageAgent]', CLEAVAGE_ONT[enzyme]); break

    # Label
    for lk in sorted(LABEL_ONT, key=len, reverse=True):
        if re.search(r'\b' + re.escape(lk) + r'\b', text_low):
            add('Characteristics[Label]', LABEL_ONT[lk]); break

    # Reduction
    for k in sorted(REDUCTION_ONT, key=len, reverse=True):
        if re.compile(_NEG + r'\b' + re.escape(k) + r'\b', re.I).search(text_low):
            add('Characteristics[ReductionReagent]', REDUCTION_ONT[k]); break

    # Alkylation
    for k in sorted(ALKYLATION_ONT, key=len, reverse=True):
        if re.compile(r'\b' + re.escape(k) + r'\b', re.I).search(text_low):
            add('Characteristics[AlkylationReagent]', ALKYLATION_ONT[k]); break

    # Modifications
    for k in sorted(MOD_ONT, key=len, reverse=True):
        if re.search(r'\b' + re.escape(k) + r'\b', text_low):
            add('Characteristics[Modification]', MOD_ONT[k])

    # Instrument
    for k in sorted(INSTRUMENT_ONT, key=len, reverse=True):
        if re.search(r'\b' + re.escape(k) + r'\b', text_low):
            add('Comment[Instrument]', INSTRUMENT_ONT[k]); break

    # Fragmentation
    for k in sorted(FRAG_ONT, key=len, reverse=True):
        if re.search(r'\b' + re.escape(k) + r'\b', text_low):
            add('Comment[FragmentationMethod]', FRAG_ONT[k])

    # Acquisition method
    for pat, val in [
        (re.compile(r'\b(dda|data[\s\-]dependent)\b', re.I), 'AC=MS:1003215;NT=DDA'),
        (re.compile(r'\b(dia|data[\s\-]independent|swath)\b', re.I), 'AC=MS:1003215;NT=DIA'),
        (re.compile(r'\b(prm|parallel\s+reaction\s+monitoring)\b', re.I), 'AC=MS:1001501;NT=PRM'),
        (re.compile(r'\b(srm|mrm)\b', re.I), 'AC=MS:1001501;NT=SRM'),
    ]:
        if pat.search(text): add('Comment[AcquisitionMethod]', val); break

    # Ionization
    if re.search(r'\b(nano[\s\-]?esi|nesi)\b', text, re.I):
        add('Comment[IonizationType]', 'AC=MS:1000398;NT=nanoESI')
    elif re.search(r'\b(electrospray|esi)\b', text, re.I):
        add('Comment[IonizationType]', 'AC=MS:1000073;NT=ESI')

    # MS2 mass analyzer
    for pat, val in [
        (re.compile(r'\b(orbitrap)\b', re.I), 'AC=MS:1000484;NT=Orbitrap'),
        (re.compile(r'\b(ion\s*trap)\b', re.I), 'AC=MS:1000264;NT=ion trap'),
        (re.compile(r'\b(tof)\b', re.I), 'AC=MS:1000084;NT=TOF'),
    ]:
        if pat.search(text): add('Comment[MS2MassAnalyzer]', val); break

    # Sex
    if re.search(r'\b(male\s+and\s+female|both\s+sexes)\b', text, re.I):
        add('Characteristics[Sex]', 'male and female')
    elif re.search(r'\b(male\s+(?:donors?|subjects?|patients?|mice|rats?))\b', text, re.I):
        add('Characteristics[Sex]', 'male')
    elif re.search(r'\b(female\s+(?:donors?|subjects?|patients?|mice|rats?))\b', text, re.I):
        add('Characteristics[Sex]', 'female')

    # Disease (gated on clinical context)
    if _CLINICAL.search(text):
        for pat, val in [
            (re.compile(r'\b(alzheimer[\s\']?s?\s+disease|ad\s+(?:patients?|brains?))\b', re.I), 'Alzheimer disease'),
            (re.compile(r'\b(parkinson[\s\']?s?\s+disease)\b', re.I), 'Parkinson disease'),
            (re.compile(r'\b(type\s+2\s+diabetes(?:\s+mellitus)?)\b', re.I), 'type 2 diabetes mellitus'),
            (re.compile(r'\b(breast\s+(?:cancer|carcinoma))\b', re.I), 'breast carcinoma'),
            (re.compile(r'\b(colorectal\s+(?:cancer|carcinoma)|colon\s+cancer)\b', re.I), 'colorectal carcinoma'),
            (re.compile(r'\b(non[\s\-]small[\s\-]cell\s+lung|nsclc)\b', re.I), 'non-small cell lung carcinoma'),
            (re.compile(r'\b(lung\s+(?:cancer|carcinoma))\b', re.I), 'lung carcinoma'),
            (re.compile(r'\b(glioblastoma|gbm)\b', re.I), 'glioblastoma'),
            (re.compile(r'\b(melanoma)\b', re.I), 'melanoma'),
            (re.compile(r'\b(prostate\s+(?:cancer|carcinoma))\b', re.I), 'prostate carcinoma'),
            (re.compile(r'\b(ovarian\s+(?:cancer|carcinoma))\b', re.I), 'ovarian carcinoma'),
            (re.compile(r'\b(covid[\s\-]?19|sars[\s\-]?cov[\s\-]?2)\b', re.I), 'COVID-19'),
            (re.compile(r'\b(osteoarthritis)\b', re.I), 'osteoarthritis'),
            (re.compile(r'\b(rheumatoid\s+arthritis)\b', re.I), 'rheumatoid arthritis'),
            (re.compile(r'\b(amyotrophic\s+lateral\s+sclerosis|als)\b', re.I), 'amyotrophic lateral sclerosis'),
            (re.compile(r'\b(healthy\s+(?:controls?|donors?|volunteers?))\b', re.I), 'normal'),
        ]:
            if pat.search(text): add('Characteristics[Disease]', val)

    # Strain
    for pat, val in [
        (re.compile(r'\b(C57BL/6J?)\b'), 'C57BL/6J'),
        (re.compile(r'\b(BALB/c)\b'), 'BALB/c'),
        (re.compile(r'\b(Sprague[\s\-]Dawley)\b', re.I), 'Sprague-Dawley'),
        (re.compile(r'\b(Wistar)\b', re.I), 'Wistar'),
    ]:
        m = pat.search(text)
        if m: add('Characteristics[Strain]', val); break

    # Genotype
    for pat, val in [
        (re.compile(r'\b(wild[\s\-]?type|wt(?:\s+cells?|\s+mice)?)\b', re.I), 'wild-type'),
        (re.compile(r'\b(knockout|knock[\s\-]out|ko(?:\s+cells?|\s+mice)?)\b', re.I), 'knockout'),
        (re.compile(r'\b(transgenic)\b', re.I), 'transgenic'),
    ]:
        if pat.search(text): add('Characteristics[Genotype]', val); break

    # DevelopmentalStage
    for pat, val in [
        (re.compile(r'\b(adult(?:s)?)\b', re.I), 'adult'),
        (re.compile(r'\b(embryo(?:nic)?)\b', re.I), 'embryo'),
        (re.compile(r'\b(fetal|fetus|foetal)\b', re.I), 'fetal'),
        (re.compile(r'\b(neonatal|newborn)\b', re.I), 'neonatal'),
    ]:
        if pat.search(text): add('Characteristics[DevelopmentalStage]', val); break

    # Numeric protocol fields
    for pat in [
        re.compile(r'(\d+)[\s\-]min(?:ute)?\s+(?:gradient|linear\s+gradient)\b', re.I),
        re.compile(r'gradient\s+(?:of\s+)?(\d+)[\s\-]?min\b', re.I),
    ]:
        m = pat.search(text)
        if m: add('Comment[GradientTime]', f'{m.group(1)} min'); break

    m = re.search(r'(\d+(?:\.\d+)?)\s*(nl|nL|µl|µL|ul|uL)\s*/\s*min', text)
    if m:
        unit = 'nL' if m.group(2).lower() == 'nl' else 'µL'
        add('Comment[FlowRateChromatogram]', f'{m.group(1)} {unit}/min')

    for pat in [
        re.compile(r'(?:precursor|ms1)\s+(?:mass\s+)?tolerance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(ppm|da)', re.I),
        re.compile(r'(\d+(?:\.\d+)?)\s*ppm\s+(?:for\s+)?(?:precursor|ms1)', re.I),
    ]:
        m = pat.search(text)
        if m:
            unit = m.group(2) if m.lastindex and m.lastindex >= 2 else 'ppm'
            add('Comment[PrecursorMassTolerance]', f'{m.group(1)} {unit}'); break

    for pat in [
        re.compile(r'(?:fragment|ms2)\s+(?:mass\s+)?tolerance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(ppm|da|mda)', re.I),
        re.compile(r'(\d+(?:\.\d+)?)\s*(da|mda)\s+(?:for\s+)?(?:fragment|ms2)', re.I),
    ]:
        m = pat.search(text)
        if m:
            unit = m.group(2) if m.lastindex and m.lastindex >= 2 else 'Da'
            add('Comment[FragmentMassTolerance]', f'{m.group(1)} {unit}'); break

    for pat in [
        re.compile(r'(?:up\s+to\s+|allowing\s+(?:up\s+to\s+)?)(\d)\s+missed\s+cleavages?', re.I),
        re.compile(r'missed\s+cleavages?\s*[=:≤]\s*(\d)', re.I),
    ]:
        m = pat.search(text)
        if m: add('Comment[NumberOfMissedCleavages]', m.group(1)); break

    # MaterialType
    if re.search(r'\b(hela|hek|jurkat|mcf7|a549)\b', text_low):
        add('Characteristics[MaterialType]', 'cell line')
    elif re.search(r'\b(tissue(?:s)?(?!\s+culture)|biopsy|tumor)\b', text, re.I):
        add('Characteristics[MaterialType]', 'tissue')
    elif re.search(r'\b(plasma|serum|urine|csf|whole\s+blood)\b', text, re.I):
        add('Characteristics[MaterialType]', 'biofluid')

    # Cap multi-value columns
    for col in ['Characteristics[OrganismPart]','Characteristics[Modification]',
                'Characteristics[Disease]']:
        if col in out: out[col] = out[col][:4]

    return dict(out)

print('Regex extractor defined.')

## 7. LLM agent

Runs AFTER regex. Only asks for columns regex can't reliably get:
CellType, CellLine, AncestryCategory, DiseaseTreatment, Specimen,
and any disease/organism/tissue that regex missed.

Two-step: extract → targeted retry for required missing cols.

In [ ]:
MAX_WORDS = 2500
MAX_FILES = 15
MAX_NEW_TOKENS = 700

# Columns regex can't reliably fill — LLM handles these
LLM_TARGET_COLS = [
    'Characteristics[CellType]',
    'Characteristics[CellLine]',
    'Characteristics[AncestryCategory]',
    'Characteristics[Specimen]',
    'Characteristics[Treatment]',
    'Characteristics[Compound]',
    'Characteristics[Age]',
    'Characteristics[BMI]',
    'Characteristics[TumorStage]',
    'Characteristics[TumorGrade]',
    # Also fill these if regex missed them
    'Characteristics[Organism]',
    'Characteristics[Disease]',
    'Characteristics[OrganismPart]',
    'Characteristics[Label]',
    'Characteristics[CleavageAgent]',
    'Comment[Instrument]',
]

SYSTEM_LLM = """You extract proteomics metadata from scientific papers.
Return ONLY a flat JSON object. Keys are SDRF column names, values are strings.
Use null if a field is not mentioned. Output ONLY JSON, no explanation."""

def run_llm(messages, max_new_tokens=MAX_NEW_TOKENS):
    torch.cuda.empty_cache()
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

def parse_json(text):
    text = re.sub(r'```json|```', '', text).strip()
    start = text.find('{')
    if start == -1: return {}
    depth = end = 0
    for i, ch in enumerate(text[start:], start):
        if ch == '{': depth += 1
        elif ch == '}': depth -= 1
        if depth == 0: end = i + 1; break
    if not end: return {}
    try: return json.loads(text[start:end])
    except:
        fixed = re.sub(r',\s*([}\]])', r'\1', text[start:end])
        try: return json.loads(fixed)
        except: return {}

def llm_extract(pub_dict, raw_files, already_filled):
    """Ask LLM only for cols regex didn't fill."""
    needed = [c for c in LLM_TARGET_COLS if c not in already_filled]
    if not needed: return {}

    text = get_text(pub_dict)
    truncated = ' '.join(text.split()[:MAX_WORDS])
    sampled = raw_files[:MAX_FILES]
    col_list = ', '.join(needed)

    user_msg = (
        f'Extract ONLY these fields: {col_list}\n\n'
        f'PAPER:\n{truncated}\n\n'
        f'RAW FILES (sample):\n' + '\n'.join(sampled) + '\n\n'
        f'Return JSON with exactly those keys. null if not found.'
    )
    raw = run_llm([
        {'role': 'system', 'content': SYSTEM_LLM},
        {'role': 'user',   'content': user_msg},
    ])
    return parse_json(raw)

def normalize_llm_output(extracted):
    """Normalize LLM extractions through ontologies."""
    normed = {}
    for col, val in extracted.items():
        if not val or str(val).strip().lower() in ('null','none','n/a','not applicable',''):
            continue
        val_s = str(val).strip()
        if 'Instrument' in col:
            n = normalize_instrument(val_s); normed[col] = n or val_s
        elif 'Organism]' in col and 'OrganismPart' not in col:
            n = normalize(val_s, ORGANISM_ONT); normed[col] = n or val_s
        elif 'OrganismPart' in col:
            n = normalize(val_s, TISSUE_ONT); normed[col] = n or val_s
        elif 'CleavageAgent' in col:
            n = normalize(val_s, CLEAVAGE_ONT); normed[col] = n or val_s
        elif 'Label' in col:
            n = normalize(val_s, LABEL_ONT); normed[col] = n or val_s
        elif 'FragmentationMethod' in col:
            n = normalize(val_s, FRAG_ONT); normed[col] = n or val_s
        elif 'Modification' in col:
            parts = [p.strip() for p in val_s.split(';') if p.strip()]
            normed[col] = [normalize(p, MOD_ONT) or p for p in parts]
        else:
            normed[col] = val_s
    return normed

print('LLM agent defined.')
print(f'LLM target cols: {len(LLM_TARGET_COLS)}')

In [ ]:
def parse_fraction(rf):
    for p in [r'[_\-]f(\d{1,3})[_\-\.]', r'[_\-]fr(\d{1,3})[_\-\.]',
              r'fraction(\d{1,3})', r'(\d{1,3})of\d+']:
        m = re.search(p, str(rf), re.I)
        if m and m.group(1).isdigit() and 1 <= int(m.group(1)) <= 200:
            return str(int(m.group(1)))
    return None

def parse_biol_replicate(rf):
    for p in [r'[_\-]biolrep[_\-]?(\d+)', r'[_\-]br(\d+)[_\-\.]',
              r'[_\-]rep(\d+)[_\-\.]', r'[_\-]r(\d{1,2})[_\-\.]']:
        m = re.search(p, str(rf), re.I)
        if m and m.group(1).isdigit() and 1 <= int(m.group(1)) <= 50:
            return str(int(m.group(1)))
    return None

def parse_label_from_filename(rf):
    rf_up = str(rf).upper()
    m = re.search(r'TMT(PRO|18|16|11|10|6|2)', rf_up)
    if m:
        plex = m.group(1)
        pmap = {'PRO':'16','18':'18','16':'16','11':'11','10':'10','6':'6','2':'2'}
        amap = {'18':'MS:1003999','16':'MS:1003998','11':'MS:1002454',
                '10':'MS:1002454','6':'MS:1002453','2':'MS:1002456'}
        p = pmap.get(plex,'6')
        return f'AC={amap[p]};NT=TMT{p}plex'
    if 'TMT' in rf_up: return 'AC=MS:1002453;NT=TMT6plex'
    if re.search(r'ITRAQ(8|4)', rf_up):
        p = re.search(r'ITRAQ(8|4)', rf_up).group(1)
        return f"AC={'MS:1002519' if p=='8' else 'MS:1001985'};NT=iTRAQ{p}plex"
    if re.search(r'SILAC|_H_|_HVY|_L_|_LGT', rf_up): return 'AC=MS:1002791;NT=SILAC'
    if re.search(r'LFQ|LABELFREE|_LF_', rf_up): return 'AC=MS:1002038;NT=label free sample'
    return None

def parse_condition(rf):
    rf_up = str(rf).upper()
    conds = {}
    if re.search(r'[_\-\.](WT|WILDTYPE)[_\-\.]', rf_up): conds['Characteristics[Genotype]'] = 'wild-type'
    elif re.search(r'[_\-\.](KO|KNOCKOUT)[_\-\.]', rf_up): conds['Characteristics[Genotype]'] = 'knockout'
    if re.search(r'[_\-\.](AD|ALZ)[_\-\.]', rf_up): conds['Characteristics[Disease]'] = 'Alzheimer disease'
    if re.search(r'[_\-\.](CTRL|CTL|HC)[_\-\.]', rf_up): conds['Characteristics[Disease]'] = 'normal'
    if re.search(r'[_\-\.](PL|PLASMA)[_\-\.]', rf_up): conds['Characteristics[OrganismPart]'] = 'NT=blood plasma;AC=UBERON:0001969'
    elif re.search(r'[_\-\.](SER|SERUM)[_\-\.]', rf_up): conds['Characteristics[OrganismPart]'] = 'NT=blood serum;AC=UBERON:0001977'
    elif re.search(r'[_\-\.](CSF)[_\-\.]', rf_up): conds['Characteristics[OrganismPart]'] = 'NT=cerebrospinal fluid;AC=UBERON:0001359'
    return conds

print('Filename parsers ready.')

## 8. Main pipeline

**Priority order (later layers only fill empty cells):**

1. PRIDE API
2. Regex from paper text
3. LLM for remaining semantic fields
4. Per-file filename parsing (overrides PXD-level for fraction/replicate/condition)
5. Majority fallback for high-frequency columns still empty

In [ ]:
from tqdm import tqdm

final_sub = pd.read_csv(SAMPLE_SUB, dtype=str).copy()
for col in target_cols:
    final_sub[col] = 'Not Applicable'

FALLBACK_THRESHOLD = 0.80

for pxd, pxd_df in tqdm(final_sub.groupby('PXD'), desc='PXDs'):
    idx       = pxd_df.index
    raw_files = pxd_to_raws[pxd]
    pub_dict  = test_docs.get(pxd, {})

    pxd_vals = defaultdict(list)   # accumulated PXD-level values

    def pxd_set(col, val):
        if val and str(val).strip().lower() not in ('not applicable','na','n/a','','null','none'):
            if val not in pxd_vals[col]:
                pxd_vals[col].append(str(val).strip())

    # ── Layer 1: PRIDE API ────────────────────────────────────────────────
    for col, vals in fetch_pride(pxd).items():
        for v in (vals or []): pxd_set(col, v)
    if PRIDE_TIMEOUT: time.sleep(0.3)

    # ── Layer 2: Regex ────────────────────────────────────────────────────
    if pub_dict:
        for col, vals in regex_extract(pub_dict).items():
            if isinstance(vals, list):
                for v in vals: pxd_set(col, v)
            else:
                pxd_set(col, vals)

    # ── Layer 3: LLM (only for cols still empty after regex) ──────────────
    if pub_dict:
        already_filled = set(pxd_vals.keys())
        try:
            llm_raw = llm_extract(pub_dict, raw_files, already_filled)
            llm_normed = normalize_llm_output(llm_raw)
            n_llm = 0
            for col, val in llm_normed.items():
                if col not in pxd_vals:  # don't overwrite regex
                    if isinstance(val, list):
                        for v in val: pxd_set(col, v)
                    else:
                        pxd_set(col, val)
                    n_llm += 1
            print(f'  {pxd}: regex={len(already_filled)} cols, LLM added {n_llm} new cols')
        except Exception as e:
            print(f'  {pxd}: LLM error: {e}')

    # ── Layer 4: Majority fallback for still-empty high-freq cols ─────────
    filled_bases = set(re.sub(r'\.\d+$','',c) for c in pxd_vals)
    for col in target_cols:
        base = re.sub(r'\.\d+$','',col)
        if base not in filled_bases and col in global_modes:
            total = sum(col_counters[col].values())
            if total > 0:
                top_freq = col_counters[col].most_common(1)[0][1] / total
                if top_freq >= FALLBACK_THRESHOLD:
                    pxd_set(col, global_modes[col])

    # Handle Modification slots
    mods = list(dict.fromkeys(pxd_vals.pop('Characteristics[Modification]', [])))
    for i, mod in enumerate(mods):
        slot = 'Characteristics[Modification]' if i==0 else f'Characteristics[Modification].{i}'
        pxd_vals[slot] = [mod]

    # ── Layer 5: Write per-file ───────────────────────────────────────────
    for i, (row_idx, raw_file) in enumerate(zip(idx, raw_files)):

        fraction = parse_fraction(raw_file)
        biol_rep = parse_biol_replicate(raw_file)
        fn_label = parse_label_from_filename(raw_file)
        fn_conds = parse_condition(raw_file)

        if fraction:
            final_sub.at[row_idx, 'Comment[FractionIdentifier]'] = fraction
        if biol_rep:
            final_sub.at[row_idx, 'Characteristics[BiologicalReplicate]'] = biol_rep
        if fn_label:
            final_sub.at[row_idx, 'Characteristics[Label]'] = fn_label
        for cond_col, cond_val in fn_conds.items():
            if cond_col in final_sub.columns:
                final_sub.at[row_idx, cond_col] = cond_val

        for col in target_cols:
            if final_sub.at[row_idx, col] != 'Not Applicable': continue
            base = re.sub(r'\.\d+$','',col)
            vals = pxd_vals.get(col) or pxd_vals.get(base) or []
            vals = [v for v in vals if str(v).strip().lower() not in ('not applicable','')]
            if vals:
                final_sub.at[row_idx, col] = vals[i % len(vals)]

# Cleanup
final_sub = final_sub.fillna('Not Applicable')
for col in target_cols:
    mask = final_sub[col].astype(str).str.strip().isin(
        ['nan','None','[]','','null','not available','TextSpan','not applicable']
    )
    final_sub.loc[mask, col] = 'Not Applicable'

# FractionIdentifier fix — if all files in a PXD have the same value '1' it's a fallback artifact
df_str = final_sub.copy()
for pxd, grp in df_str.groupby('PXD'):
    fracs = grp['Comment[FractionIdentifier]'].unique()
    if len(fracs) == 1 and str(fracs[0]).strip() in ('1','1.0'):
        final_sub.loc[grp.index, 'Comment[FractionIdentifier]'] = 'Not Applicable'

final_sub.to_csv(OUT_PATH, index=False)
print(f'\nSaved → {OUT_PATH}')
print(f'Shape : {final_sub.shape}')

In [ ]:
label_cols = [c for c in final_sub.columns
              if c not in ('ID','PXD','Raw Data File','Usage')]
rows = [(c,(final_sub[c]!='Not Applicable').sum()) for c in label_cols]
rows.sort(key=lambda x: -x[1])

print(f'{"Column":<55} {"Filled":>7} {"Pct":>6}')
print('-'*72)
for col, n in rows:
    if n > 0:
        source = ''
        print(f'{col:<55} {n:>7} {n/len(final_sub)*100:>5.1f}%')

zero = sum(1 for _,n in rows if n==0)
print(f'\nFilled : {len(rows)-zero} / {len(rows)} columns')
print(f'Rows   : {len(final_sub)}')

# Key column spot check
print('\nKey column values:')
for col in ['Comment[Instrument]','Characteristics[Organism]',
            'Characteristics[CleavageAgent]','Characteristics[Modification]',
            'Comment[FractionIdentifier]','Characteristics[BiologicalReplicate]']:
    if col in final_sub.columns:
        vc = final_sub[col].value_counts().head(3)
        print(f'  {col}:')
        for v, n in vc.items():
            print(f'    {str(v)[:60]}: {n}')